# One-region noise simulation


No inputs, just noise.


$$
\begin{aligned}
\nu_e^{\mathrm{ext},(r)} =& \boxed{c_0} + k_{\mathrm{local}}\nu_e + \boxed{w_{\mathrm{noise}}x_{\mathrm{OU}}} + \boxed{\nu^{\mathrm{drive}}_{e\to(r)}}\\
\nu_i^{\mathrm{ext},(r)} =& k_{\mathrm{local}}\nu_i + \nu^{\mathrm{drive}}_{i\to(r)}.
\end{aligned}
$$


## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path


# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from aux.environment import resolve_directory
from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import build_single_region_simulator
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results


Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Simulation values.
DURATION_MS = 2000.0
DT_MS = 0.5

OUTPUT_DTYPE = "float32"
OUTPUT_STATES = ["E"]
# El rango del modelo sugiere que la tasa E se expresa en kHz.
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L351
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L671
STATE_UNITS = {"E": "kHz"}

# El rango y la suma con Fe sin conversión sugieren kHz para la entrada externa.
# Cálculo de Fe: https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L307
# Suma con external_input: https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L429
# La tabla del artículo y la documentación del modelo indican Hz.
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L116
# https://link.springer.com/article/10.1007/s10827-017-0668-2/tables/1
BASELINE_KHZ = 0.001


In [ ]:
# 1.1) Noise grid. The integrator noise and its seed stay fixed.
# Los parámetros de ruido no figuran en la tabla citada; los valores por defecto
# del código difieren de los valores de referencia explorados aquí.
WEIGHT_NOISE_VALUES = [0.0, 2e-4, 2e-3, 1e-2]
TAU_OU_VALUES = [0.5, 2.0, 8.0]
NOISE_SEED = 42

NOISE = {
    "name": "Additive",
    "params": {"nsig": [0, 0, 0, 0, 1]},
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"

# Parametros sacados 
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L60
# Otras fuentes de parametros por defecto:
# https://github.com/the-virtual-brain/tvb-root/blob/8e766d68622fa2f6b057119ac5a1cf3652ea6f89/tvb_library/tvb/simulator/models/zerlaut.py#L148
# https://link.springer.com/article/10.1007/s10827-017-0668-2/tables/1

MODEL_PARAMETERS = {
    # Población
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Celulares: propiedades eléctricas
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Celulares: parámetros de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 0.0, 
    "tau_w_i": 1.0, #!No puede ser 0 como en la tabla

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    # Celulares: escala de tiempo
    "T": 10.0,

    # Parámetro del modelo de segundo orden; las simulaciones de este cuaderno usan primer orden.
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Coeficientes de la función de transferencia: phenomenological threshold coefficients
    "P_e": [-0.04983106, 0.005063550882777035, -0.023470121807314552, 0.0022951513725067503, -0.0004105302652029825, 0.010547051343547399, -0.03659252821136933, 0.007437487505797858, 0.001265064721846073, -0.04072161294490446],
    "P_i": [-0.05149122024209484, 0.004003689190271077, -0.008352013668528155, 0.0002414237992765705, -0.0005070645080016026, 0.0014345394104282397, -0.014686689498949967, 0.004502706285435741, 0.0028472190352532454, -0.015357804594594548],

}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


The constant baseline is assigned to the model parameter `external_input_ex_ex`. `weight_noise` scales the Ornstein-Uhlenbeck contribution to the external input; `tau_OU` changes its time scale. The stochastic integrator drives only `ou_drift`, with the same random sequence in every grid cell.


## Stochastic Simulation


### Run simulation


In [ ]:
results = {}
for tau_OU in TAU_OU_VALUES:
    for weight_noise in WEIGHT_NOISE_VALUES:
        parameters = MODEL_PARAMETERS | {
            "external_input_ex_ex": BASELINE_KHZ,
            "weight_noise": weight_noise,
            "tau_OU": tau_OU,
        }

        integrator = build_integrator(DT_MS, integrator_class="HeunStochastic",
                                      noise_config=NOISE)
        
        model = build_model(
            parameters, INTEGER_PARAMETERS,
            model_class="ZerlautAdaptationFirstOrder",
            variable_of_interest=OUTPUT_STATES,
        )
        
        simulator = build_single_region_simulator(model, [], integrator)

        integrator.noise.random_stream.seed(NOISE_SEED)
        time_sim, raw = simulator.run(n_steps=round(DURATION_MS / DT_MS))[0]
        
        states = {"E": raw[:, :, 0, 0]}
        metadata = ResultMetadata(
            simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
            model=ModelMetadata(
                name=model, parameters=parameters,
                integer_parameters=list(INTEGER_PARAMETERS),
                output_variables=OUTPUT_STATES, state_units=STATE_UNITS,
            ),
            stimulus={},
        )

        results[(tau_OU, weight_noise)] = (time_sim, states, metadata)
        
        pprint(f"tau_OU={tau_OU:g} ms, weight_noise={weight_noise:g}: "
               f"E standard deviation={states['E'].std():.5f} kHz")


### Plot results


In [ ]:
fig, axes = plt.subplots(
    len(TAU_OU_VALUES), len(WEIGHT_NOISE_VALUES),
    figsize=(4.5 * len(WEIGHT_NOISE_VALUES), 2.8 * len(TAU_OU_VALUES)),
    constrained_layout=True,
)


plotted_axes = []
for row, tau_OU in enumerate(TAU_OU_VALUES):
    for column, weight_noise in enumerate(WEIGHT_NOISE_VALUES):
        time_sim, states, metadata = results[(tau_OU, weight_noise)]
        _, panel_axes = plot_results(
            time_sim, states, metadata, show_stimulus=False,
            plot_states=["E"], ax=axes[row, column],
        )
        panel_axes[0].set_title(f"tau_OU={tau_OU:g} ms; weight_noise={weight_noise:g}")
        plotted_axes.append(panel_axes[0])

for axis in plotted_axes:
    axis.set_xlim(0, DURATION_MS)
fig.suptitle(f"E with constant external_input_ex_ex = {BASELINE_KHZ:g} kHz")
plt.show()
